# NB09 최종성의 확률적 성격 — 「확인 수」의 뜻

『법률가를 위한 블록체인 — 기술과 경제학으로 읽기』의 장별 노트북입니다. 본문의 코드와 같습니다.

- 모든 코드는 원리를 눈으로 보기 위한 **장난감 구현**입니다. 실제 자산·실제 개인키·실제 거래에 쓰지 마십시오(실사용 금지).
- 외부 API를 부르지 않습니다. 네트워크 없이 실행됩니다. 동봉 스냅샷 데이터가 필요한 셀은 그 데이터를 셀 안에 적어 두었습니다.
- 표준 라이브러리만 씁니다(Python 3.13에서 실행·확인). 설치 셀은 없습니다.
- 출력은 엮은이가 실행한 결과를 그대로 담아 두었습니다. 다시 실행하면 같은 값이 나와야 합니다(난수는 시드를 고정했습니다).

In [ ]:
# NB09-1: 백서 §11 의 식 — C 코드를 그대로 옮겨 논문 표(q=0.1, 0.3)를 재현한다
import math

def nakamoto_P(q: float, z: int) -> float:
    """P = 1 - Σ_{k=0}^{z} λ^k e^{-λ}/k! · (1 - (q/p)^{z-k}),  λ = z·q/p   (Nakamoto 2008, p.7)"""
    p = 1.0 - q
    lam = z * (q / p)
    total = 1.0
    for k in range(z + 1):
        poisson = math.exp(-lam)
        for i in range(1, k + 1):
            poisson *= lam / i
        total -= poisson * (1 - (q / p) ** (z - k))
    return total

def gamblers_ruin(q: float, z: int) -> float:
    """§11 첫 식 q_z: 공격자가 z 블록 뒤에서 언젠가 따라잡을 확률(포아송 진척을 무시한 값)"""
    p = 1.0 - q
    return 1.0 if p <= q else (q / p) ** z

for q, zs in [(0.1, range(0, 11)), (0.3, range(0, 51, 5))]:
    print(f"q={q}")
    for z in zs:
        print(f"  z={z:<3} P={nakamoto_P(q, z):.7f}   (q_z 도박꾼 파산만 = {gamblers_ruin(q, z):.7f})")

q=0.1
  z=0   P=1.0000000   (q_z 도박꾼 파산만 = 1.0000000)
  z=1   P=0.2045873   (q_z 도박꾼 파산만 = 0.1111111)
  z=2   P=0.0509779   (q_z 도박꾼 파산만 = 0.0123457)
  z=3   P=0.0131722   (q_z 도박꾼 파산만 = 0.0013717)
  z=4   P=0.0034552   (q_z 도박꾼 파산만 = 0.0001524)
  z=5   P=0.0009137   (q_z 도박꾼 파산만 = 0.0000169)
  z=6   P=0.0002428   (q_z 도박꾼 파산만 = 0.0000019)
  z=7   P=0.0000647   (q_z 도박꾼 파산만 = 0.0000002)
  z=8   P=0.0000173   (q_z 도박꾼 파산만 = 0.0000000)
  z=9   P=0.0000046   (q_z 도박꾼 파산만 = 0.0000000)
  z=10  P=0.0000012   (q_z 도박꾼 파산만 = 0.0000000)
q=0.3
  z=0   P=1.0000000   (q_z 도박꾼 파산만 = 1.0000000)
  z=5   P=0.1773523   (q_z 도박꾼 파산만 = 0.0144583)
  z=10  P=0.0416605   (q_z 도박꾼 파산만 = 0.0002090)
  z=15  P=0.0101008   (q_z 도박꾼 파산만 = 0.0000030)
  z=20  P=0.0024804   (q_z 도박꾼 파산만 = 0.0000000)
  z=25  P=0.0006132   (q_z 도박꾼 파산만 = 0.0000000)
  z=30  P=0.0001522   (q_z 도박꾼 파산만 = 0.0000000)
  z=35  P=0.0000379   (q_z 도박꾼 파산만 = 0.0000000)
  z=40  P=0.0000095   (q_z 도박꾼 파산만 = 0.0000000)
  z=45  P=0.0000024   (q_z 도

In [ ]:
# NB09-2: 백서 8쪽 「P < 0.1%」 — 공격자 점유율 q 마다 필요한 최소 확인 수 z
def min_z(P_func, q, threshold=0.001):
    z = 0
    while P_func(q, z) > threshold:
        z += 1
    return z

print("q      z (Nakamoto 식)")
for q in (0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45):
    print(f"{q:.2f}   {min_z(nakamoto_P, q):>4}")

q      z (Nakamoto 식)
0.10      5
0.15      8
0.20     11
0.25     15
0.30     24
0.35     41
0.40     89
0.45    340


In [ ]:
# NB09-3: 정확식 — 음이항 유한합으로 계산해 Rosenfeld 표 1·Grunspan–Pérez-Marco 표 1과 대조 (외부 패키지 없음)
from math import lgamma, log, exp

def exact_P(q: float, z: int) -> float:
    """정직 체인이 z 블록을 캐는 동안 공격자 블록 수 k ~ NegBin; k<z 면 (q/p)^(z-k) 로 따라잡음, k>=z 면 이미 따라잡음.
    이항계수가 커지므로 로그로 계산한다: log C(k+z-1, k) = lgamma(k+z) - lgamma(k+1) - lgamma(z)."""
    if z == 0: return 1.0
    p = 1.0 - q
    if q >= p: return 1.0
    caught, mass = 0.0, 0.0
    for k in range(z):
        log_pk = lgamma(k + z) - lgamma(k + 1) - lgamma(z) + z * log(p) + k * log(q)
        pk = exp(log_pk)
        mass += pk
        caught += pk * (q / p) ** (z - k)
    return caught + (1.0 - mass)

print("Rosenfeld(2014) 표 1 의 q=10% 행: 20%, 5.600%, 1.712%, 0.546%, 0.178%, 0.059%, 0.020%, 0.007%, 0.002%, 0.001%")
print("이 책의 유한합        q=10% 행:", ", ".join(f"{100*exact_P(0.10, n):.3f}%" for n in range(1, 11)))
print()
print("Grunspan–Pérez-Marco(2018) 표 1: P<0.1% 에 필요한 z —  q: 0.10 0.15 0.20 0.25 0.30 0.35 0.40 0.45")
print("  논문의 z(정확식)      : 6 9 13 20 32 58 133 539")
print("  이 책의 유한합        :", " ".join(str(min_z(exact_P, q)) for q in (0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45)))
print("  논문의 z_SN(나카모토) : 5 8 11 15 24 41 81 340   ← q=0.40 의 81 은 백서 8쪽·위 NB09-2 의 89 와 다르다(아래 본문)")
print()
print(f"{'q':>5} {'z':>4} {'Nakamoto':>10} {'정확식':>10} {'비율':>6}")
for q, z in ((0.1, 1), (0.1, 3), (0.1, 6), (0.3, 5), (0.3, 10), (0.3, 24), (0.45, 40), (0.45, 340)):
    a, b = nakamoto_P(q, z), exact_P(q, z)
    print(f"{q:>5.2f} {z:>4} {a:>10.7f} {b:>10.7f} {b/a:>6.2f}")

Rosenfeld(2014) 표 1 의 q=10% 행: 20%, 5.600%, 1.712%, 0.546%, 0.178%, 0.059%, 0.020%, 0.007%, 0.002%, 0.001%
이 책의 유한합        q=10% 행: 20.000%, 5.600%, 1.712%, 0.546%, 0.178%, 0.059%, 0.020%, 0.007%, 0.002%, 0.001%

Grunspan–Pérez-Marco(2018) 표 1: P<0.1% 에 필요한 z —  q: 0.10 0.15 0.20 0.25 0.30 0.35 0.40 0.45
  논문의 z(정확식)      : 6 9 13 20 32 58 133 539
  이 책의 유한합        : 6 9 13 20 32 58 133 539
  논문의 z_SN(나카모토) : 5 8 11 15 24 41 81 340   ← q=0.40 의 81 은 백서 8쪽·위 NB09-2 의 89 와 다르다(아래 본문)

    q    z   Nakamoto        정확식     비율
 0.10    1  0.2045873  0.2000000   0.98
 0.10    3  0.0131722  0.0171200   1.30
 0.10    6  0.0002428  0.0005914   2.44
 0.30    5  0.1773523  0.1976173   1.11
 0.30   10  0.0416605  0.0651067   1.56
 0.30   24  0.0008106  0.0040007   4.94
 0.45   40  0.3454201  0.3713860   1.08
 0.45  340  0.0009947  0.0089679   9.02


In [ ]:
# NB09-4: 몬테카를로 — 식을 믿지 말고 경주를 직접 해 본다 (시드 고정)
import random

def simulate(q: float, z: int, trials: int, seed: int = 20261025) -> float:
    """정직 체인이 z 블록을 캘 때까지 공격자도 캔다(블록마다 확률 q). 그 뒤 격차가 0 이하가 되면 성공, z+200 넘게 벌어지면 실패로 절단."""
    rng = random.Random(seed)
    p = 1.0 - q
    wins = 0
    for _ in range(trials):
        honest, attacker = 0, 0
        while honest < z:                        # 상인이 z 확인을 기다리는 동안
            if rng.random() < q: attacker += 1
            else: honest += 1
        gap = honest - attacker
        while 0 < gap < z + 200:                 # 그 뒤의 경주
            gap += -1 if rng.random() < q else 1
        if gap <= 0: wins += 1
    return wins / trials

for q, z, n in ((0.1, 3, 200_000), (0.1, 6, 200_000), (0.3, 10, 50_000)):
    mc, ex, na = simulate(q, z, n), exact_P(q, z), nakamoto_P(q, z)
    se = (ex * (1 - ex) / n) ** 0.5
    print(f"q={q} z={z:>2} n={n:>7,}: 몬테카를로 {mc:.5f} | 정확식 {ex:.5f} | 나카모토 {na:.5f} | MC 표준오차 ≈ {se:.5f}")

q=0.1 z= 3 n=200,000: 몬테카를로 0.01728 | 정확식 0.01712 | 나카모토 0.01317 | MC 표준오차 ≈ 0.00029
q=0.1 z= 6 n=200,000: 몬테카를로 0.00061 | 정확식 0.00059 | 나카모토 0.00024 | MC 표준오차 ≈ 0.00005
q=0.3 z=10 n= 50,000: 몬테카를로 0.06414 | 정확식 0.06511 | 나카모토 0.04166 | MC 표준오차 ≈ 0.00110


In [ ]:
# NB09-5: 절대 해시율 불변성 — 식에는 점유율 q 만 들어간다
print("공격자 해시율 a, 정직 해시율 h  →  q = a/(a+h),  P(z=6)")
for a, h in ((1, 9), (1e6, 9e6), (1e20, 9e20), (3e20, 7e20)):
    q = a / (a + h)
    print(f"  a={a:.0e} H/s, h={h:.0e} H/s → q={q:.2f}, 정확식 P(6)={exact_P(q, 6):.6f}, 나카모토 P(6)={nakamoto_P(q, 6):.6f}")

공격자 해시율 a, 정직 해시율 h  →  q = a/(a+h),  P(z=6)
  a=1e+00 H/s, h=9e+00 H/s → q=0.10, 정확식 P(6)=0.000591, 나카모토 P(6)=0.000243
  a=1e+06 H/s, h=9e+06 H/s → q=0.10, 정확식 P(6)=0.000591, 나카모토 P(6)=0.000243
  a=1e+20 H/s, h=9e+20 H/s → q=0.10, 정확식 P(6)=0.000591, 나카모토 P(6)=0.000243
  a=3e+20 H/s, h=7e+20 H/s → q=0.30, 정확식 P(6)=0.156450, 나카모토 P(6)=0.132111
